# catboost_postprocess_v1.ipynb
**仅后处理**（不重训底模、无需 LGBM）：
- 基于 OOF 学习 **分段等距回归（Isotonic）校正** 与 **Logistic 二级矫正**（使用 `logit(p)` + 稳定元特征）。
- 产出 3 套测试提交：`segment_isotonic`、`rank_blend(segment_iso, raw)`、`stack_lr`。
- 兼容你的初版变量命名：若内存里已有 `tr/models/features/te_aa/te_ab` 则直接复用；否则回退读取常见文件名。

In [7]:

# =============== 配置 ===============
OUT_DIR = "outputs_postprocess_v1"
OOF_PATH_CANDIDATES = [
    "outputs/oof_predictions.csv",
    "oof_predictions.csv",
    "outputs_timeaware_v2/oof_timeaware.csv",
    "outputs_v1/oof_predictions.csv",
    "outputs_v1plus_lgbm/oof_cat.csv"  # 兼容旧命名
]
TEST_AA_PATH_CANDIDATES = [
    "outputs/test_aa_pred_cat.csv",
    "test_pred_catboost.csv",
    "outputs_timeaware_v2/test_aa_pred_timeaware.csv"
]
TEST_AB_PATH_CANDIDATES = [
    "outputs/test_ab_pred_cat.csv",
    "outputs_timeaware_v2/test_ab_pred_timeaware.csv"
]

MIN_SEG_SAMPLES = 300       # 分段最小样本数；不足回退为全局模型
TIME_BIN_QUANTILES = 5      # issue_time_days 分箱数量（等频）
RANDOM_STATE = 1337


In [8]:

import os, glob, numpy as np, pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

os.makedirs(OUT_DIR, exist_ok=True)

# --------- 解析/回退：训练集、特征、模型、OOF与测试预测 ---------
def _find_first(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def resolve_train_df():
    for name in ['tr','train','df_train','train_df']:
        if name in globals():
            df = globals()[name]
            if isinstance(df, pd.DataFrame) and {'id','label'}.issubset(df.columns):
                return df
    if os.path.exists('train.csv'):
        return pd.read_csv('train.csv')
    raise RuntimeError("未找到训练集 DataFrame，也无法读取 train.csv")

def resolve_features(df):
    if 'features' in globals():
        lst = globals()['features']
        if all(c in df.columns for c in lst):
            return list(lst)
    # 回退：去掉 id/label
    return [c for c in df.columns if c not in ('id','label')]

def resolve_models():
    for name in ['models','cat_models','cb_models']:
        if name in globals():
            m = globals()[name]
            if isinstance(m, (list, tuple)) and len(m)>0:
                return m
    return None  # 后续用 CSV 预测回退

def resolve_oof(tr_df):
    # 1) 内存变量
    for name in ['oof','oof_pred','oof_proba','oof_predictions']:
        if name in globals():
            arr = np.asarray(globals()[name], dtype=float).reshape(-1)
            if len(arr) == len(tr_df):
                return arr
    # 2) CSV 候选
    p = _find_first(OOF_PATH_CANDIDATES)
    if p:
        df = pd.read_csv(p)
        # 常见列名兼容
        for col in ['oof_pred','oof','prob','pred','oof_cat','oof_proba','oof_timeaware','oof_blend']:
            if col in df.columns:
                # 尝试与 id 对齐
                if 'id' in df.columns:
                    m = tr_df[['id']].merge(df[['id',col]], on='id', how='left')
                    if m[col].notna().sum() > 0:
                        return m[col].fillna(m[col].mean()).values
                return df[col].values
    raise RuntimeError("未找到 OOF 预测（既没有内存变量，也没有常见 CSV）")

def resolve_test_df(tag):
    # 优先内存
    if tag=='aa':
        for name in ['te_aa','test_aa','df_test_aa']:
            if name in globals() and isinstance(globals()[name], pd.DataFrame):
                return globals()[name]
    if tag=='ab':
        for name in ['te_ab','test_ab','df_test_ab','test']:
            if name in globals() and isinstance(globals()[name], pd.DataFrame):
                return globals()[name]
    # 回退 CSV
    path = _find_first(TEST_AA_PATH_CANDIDATES if tag=='aa' else TEST_AB_PATH_CANDIDATES)
    if path:
        # 仅有预测CSV时，我们另行读取 test*.csv 以拿到 id 与元特征
        # 这里尝试多名
        raw_path = 'testaa.csv' if tag=='aa' else 'testab.csv'
        if os.path.exists(raw_path):
            return pd.read_csv(raw_path)
    return None

def resolve_test_pred(tag, models, features, cat_cols):
    # 如果内存中已有 test 预测，就直接算；否则尝试读取 CSV
    if tag=='aa':
        for name in ['pred_test_aa','testaa_pred','pred_aa']:
            if name in globals():
                return np.asarray(globals()[name], dtype=float).reshape(-1)
    if tag=='ab':
        for name in ['pred_test_ab','testab_pred','pred_ab']:
            if name in globals():
                return np.asarray(globals()[name], dtype=float).reshape(-1)
    # CSV 读取
    p = _find_first(TEST_AA_PATH_CANDIDATES if tag=='aa' else TEST_AB_PATH_CANDIDATES)
    if p:
        df = pd.read_csv(p)
        for col in ['prob','pred','score']:
            if col in df.columns:
                return df[col].values
    # 还不行：如果有模型与 test_df，就用模型现算
    df = resolve_test_df(tag)
    if df is not None and models is not None and features is not None and cat_cols is not None:
        from catboost import Pool
        X = df[features].copy()
        pool = Pool(X, cat_features=[X.columns.get_loc(c) for c in cat_cols if c in X.columns])
        preds = np.mean([m.predict_proba(pool)[:,1] for m in models], axis=0)
        return preds
    return None


In [9]:

# --------- 元特征构造（仅用于后处理；不影响底模）---------
def build_meta_features(df):
    meta = pd.DataFrame(index=df.index)
    # grade / term / has_stm / level
    for c in ['grade','term','has_stm','level']:
        if c in df.columns:
            meta[c] = df[c].astype(str)
    # 时间分箱（issue_time_days 越大越旧，按等频分箱）
    if 'issue_time_days' in df.columns:
        try:
            meta['time_bin'] = pd.qcut(df['issue_time_days'], q=TIME_BIN_QUANTILES, duplicates='drop').astype(str)
        except Exception:
            meta['time_bin'] = 'ALL'
    else:
        meta['time_bin'] = 'ALL'
    return meta

def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps)
    return np.log(p/(1-p))

def to_rank01(x):
    r = pd.Series(x).rank(method='average').values
    return (r - r.min()) / (r.max() - r.min() + 1e-12)


In [10]:

# --------- 方案A：分段 Isotonic 回归（会改变跨段排序）---------
def fit_segment_isotonic(y, p, meta_df, min_samples=300, seg_cols=('grade','has_stm')):
    seg_key = meta_df[list(seg_cols)].astype(str).agg('|'.join, axis=1)
    models = {}
    # 全局备用
    global_iso = IsotonicRegression(out_of_bounds='clip')
    global_iso.fit(p, y)
    for k, idx in pd.Series(range(len(y))).groupby(seg_key):
        idx = idx.values
        if len(idx) < min_samples:
            continue
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(p[idx], y[idx])
        models[k] = iso
    return models, global_iso

def apply_segment_isotonic(p, meta_df, models_map, global_iso, seg_cols=('grade','has_stm')):
    seg_key = meta_df[list(seg_cols)].astype(str).agg('|'.join, axis=1)
    out = np.zeros_like(p, dtype=float)
    for k, idx in pd.Series(range(len(p))).groupby(seg_key):
        idx = idx.values
        iso = models_map.get(k, global_iso)
        out[idx] = iso.predict(p[idx])
    return out


In [11]:

# --------- 方案B：Logistic 二级（使用 logit(p) + 元特征 one-hot）---------
def fit_stack_lr(y, p, meta_df):
    X = pd.DataFrame({'logit_p': to_logit(p)})
    # one-hot 元特征（稳妥）
    cat_cols = [c for c in meta_df.columns]
    X = pd.concat([X, pd.get_dummies(meta_df[cat_cols], drop_first=True)], axis=1)
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),  # 稀疏矩阵友好
        ("lr", LogisticRegression(C=1.0, solver="lbfgs", max_iter=2000))
    ])
    pipe.fit(X, y)
    return pipe, X.columns.tolist()

def apply_stack_lr(p, meta_df, pipe, train_cols):
    X = pd.DataFrame({'logit_p': to_logit(p)})
    cat_cols = [c for c in meta_df.columns]
    X = pd.concat([X, pd.get_dummies(meta_df[cat_cols], drop_first=True)], axis=1)
    # 对齐列
    for c in train_cols:
        if c not in X.columns:
            X[c] = 0
    X = X[train_cols]
    return pipe.predict_proba(X)[:,1]


In [13]:

# ================== 主流程：读取对象 -> 训练后处理 -> 评估 OOF -> 生成 test 提交 ==================
# 解析基础对象
tr = resolve_train_df()
features = resolve_features(tr)
models = resolve_models()

# OOF 与元特征
oof_base = resolve_oof(tr)
meta_tr = build_meta_features(tr)
y_true = tr['label'].values.astype(int)

print("[INFO] OOF base AUC =", roc_auc_score(y_true, oof_base))

# --- 方案A：分段 Isotonic ---
iso_models, iso_global = fit_segment_isotonic(y_true, oof_base, meta_tr, min_samples=MIN_SEG_SAMPLES, seg_cols=('grade','has_stm'))
oof_iso = apply_segment_isotonic(oof_base, meta_tr, iso_models, iso_global, seg_cols=('grade','has_stm'))
print("[A] Segment Isotonic OOF AUC =", roc_auc_score(y_true, oof_iso))

# Rank 融合（raw vs iso）
best_w, best_auc = 0.5, -1
oof_raw_r = to_rank01(oof_base)
oof_iso_r = to_rank01(oof_iso)
for w in np.linspace(0,1,21):
    blend = w*oof_raw_r + (1-w)*oof_iso_r
    auc = roc_auc_score(y_true, blend)
    if auc > best_auc:
        best_auc, best_w = auc, float(w)
print(f"[A-rank] best_w={best_w:.2f} | OOF AUC={best_auc:.6f}")

# --- 方案B：Stack LR（logit(p)+meta）---
stack_model, stack_cols = fit_stack_lr(y_true, oof_base, meta_tr)
oof_stack = stack_model.predict_proba(pd.concat([pd.DataFrame({'logit_p': to_logit(oof_base)}), pd.get_dummies(meta_tr, drop_first=True)], axis=1)[stack_cols])[:,1]
print("[B] Stack LR OOF AUC =", roc_auc_score(y_true, oof_stack))

# ========= 输出测试集三套提交 =========
def save_sub(df, prob, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pd.DataFrame({'id': df['id'], 'prob': prob}).to_csv(path, index=False, encoding='utf-8')
    print("[SAVE]", path)

for tag in ['aa','ab']:
    df_te = resolve_test_df(tag)
    if df_te is None:
        print(f"[INFO] 无 {tag} 测试集，跳过生成。"); 
        continue
    # 基础预测（优先内存模型计算；否则读取你之前导出的 cat 提交）
    p_base = resolve_test_pred(tag, models, features, [c for c in ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade'] if c in features])
    if p_base is None:
        print(f"[WARN] 未能解析 {tag} 的基础预测，跳过。"); 
        continue
    meta_te = build_meta_features(df_te)

    # A) 分段 isotonic
    p_iso = apply_segment_isotonic(p_base, meta_te, iso_models, iso_global, seg_cols=('grade','has_stm'))
    save_sub(df_te, p_iso, os.path.join(OUT_DIR, f"test_{tag}_pred_segment_isotonic.csv"))

    # A-rank) rank 融合
    p_rank_blend = best_w * to_rank01(p_base) + (1-best_w) * to_rank01(p_iso)
    save_sub(df_te, p_rank_blend, os.path.join(OUT_DIR, f"test_{tag}_pred_rankblend_raw_iso.csv"))

    # B) stack LR
    p_stack = apply_stack_lr(p_base, meta_te, stack_model, stack_cols)
    save_sub(df_te, p_stack, os.path.join(OUT_DIR, f"test_{tag}_pred_stack_lr.csv"))


[INFO] OOF base AUC = 0.6640302545603807


KeyError: "None of [Index(['grade', 'has_stm'], dtype='object')] are in the [columns]"